In [1]:
# Cell 1 — Imports and paths
import pickle
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Project paths (notebook lives in notebooks/, so .. is the project root)
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"\nContents of data/:")
for p in sorted(DATA_DIR.iterdir()):
    kind = "dir" if p.is_dir() else f"{p.stat().st_size:>10,} bytes"
    print(f"  {p.name:<40}  {kind}")

Project root: /Users/mac/Desktop/semantic_graph_project
Data directory exists: True

Contents of data/:
  .DS_Store                                      6,148 bytes
  processed                                 dir
  raw                                       dir


In [2]:
# Cell 2 — Look inside raw/ and processed/
for subdir in ["raw", "processed"]:
    d = DATA_DIR / subdir
    print(f"=== data/{subdir}/ ===")
    if not d.exists():
        print("  (does not exist)")
        continue
    for p in sorted(d.iterdir()):
        if p.name == ".DS_Store":
            continue
        kind = "dir" if p.is_dir() else f"{p.stat().st_size:>12,} bytes"
        print(f"  {p.name:<50}  {kind}")
    print()

=== data/raw/ ===
  SWOW-EN18                                           dir
  all ldt subs_all trials3.xlsx                        110,512,713 bytes
  items_spreadsheet.xls                                  2,027,008 bytes
  ldt word item analysis.xlsx                              823,236 bytes

=== data/processed/ ===
  swow_en_graph.pkl                                      9,657,069 bytes



In [3]:
# Cell 3 — Load the Phase 1 graph
GRAPH_PATH = DATA_DIR / "processed" / "swow_en_graph.pkl"

with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

print(f"Type:         {type(G).__name__}")
print(f"Directed:     {G.is_directed()}")
print(f"Nodes:        {G.number_of_nodes():,}")
print(f"Edges:        {G.number_of_edges():,}")

# Confirm the edge weight attribute by inspecting a sample edge
sample_u, sample_v = next(iter(G.edges()))
print(f"\nSample edge: {sample_u} -> {sample_v}")
print(f"Edge attributes: {dict(G[sample_u][sample_v])}")

# Quick check that the ocean spot-check from Phase 1 still works
if "ocean" in G:
    neighbors = sorted(G["ocean"].items(),
                       key=lambda kv: kv[1].get("R123.Strength", 0),
                       reverse=True)[:6]
    print(f"\nTop 6 neighbors of 'ocean':")
    for v, attrs in neighbors:
        w = attrs.get("R123.Strength")
        print(f"  ocean -> {v:<12}  weight = {w:.4f}" if w is not None else f"  ocean -> {v}")

Type:         DiGraph
Directed:     True
Nodes:        24,657
Edges:        279,410

Sample edge: a -> abc
Edge attributes: {'R123.Strength': 0.007407407407407408}

Top 6 neighbors of 'ocean':
  ocean -> sea           weight = 0.1554
  ocean -> water         weight = 0.1250
  ocean -> blue          weight = 0.0845
  ocean -> waves         weight = 0.0541
  ocean -> vast          weight = 0.0507
  ocean -> salt          weight = 0.0372


In [4]:
# Cell 4 — Discover the structure of items_spreadsheet.xls
SPP_PATH = DATA_DIR / "raw" / "items_spreadsheet.xls"

xls = pd.ExcelFile(SPP_PATH)
print(f"File: {SPP_PATH.name}")
print(f"Number of sheets: {len(xls.sheet_names)}")
print(f"\nSheets:")
for i, name in enumerate(xls.sheet_names):
    # Peek at the shape of each sheet so we can identify the data sheet
    df = pd.read_excel(SPP_PATH, sheet_name=name, nrows=0)  # just the header row
    n_cols = len(df.columns)
    # Get row count cheaply by reading just the first column
    n_rows = len(pd.read_excel(SPP_PATH, sheet_name=name, usecols=[0]))
    print(f"  [{i}] {name!r:<35}  {n_rows:>6,} rows x {n_cols:>3} cols")

File: items_spreadsheet.xls
Number of sheets: 7

Sheets:
  [0] '1st associate definitions'              44 rows x   2 cols
  [1] 'first associate'                     1,665 rows x  45 cols
  [2] 'other associate definitions'            25 rows x   2 cols
  [3] 'other associate'                     1,665 rows x  29 cols
  [4] 'unrelated definitions'                   8 rows x   2 cols
  [5] 'unrelated pairs'                     1,665 rows x  16 cols
  [6] 'hutchison (2003) classification'        15 rows x   3 cols


In [7]:
# Cell 5 — Load the 'first associate' sheet and inspect columns
spp_raw = pd.read_excel(SPP_PATH, sheet_name="first associate")

print(f"Shape: {spp_raw.shape}")
print(f"\nAll 45 columns:")
for i, col in enumerate(spp_raw.columns):
    print(f"  [{i:>2}] {col}")

Shape: (1665, 45)

All 45 columns:
  [ 0] prime_first associate
  [ 1] f_subtl freq
  [ 2] f_log subtl freq
  [ 3] f_length
  [ 4] f_Log Hal
  [ 5] f_orthoN
  [ 6] f_freq_greater
  [ 7] f_BG_SUM
  [ 8] f_BG_mean
  [ 9] f_POS
  [10] f_ldt_RT_ELP
  [11] f_ldt_Z_ELP
  [12] f_ldt_acc_ELP
  [13] f_name_RT_ELP
  [14] f_name_Z_ELP
  [15] f_name_acc_ELP
  [16] TARGET
  [17] t_length
  [18] t_Log Hal
  [19] t_subtl freq
  [20] t_log subtl freq
  [21] t_orthoN
  [22] t_freq_greater
  [23] t_BG_SUM
  [24] t_BG_mean
  [25] t_POS
  [26] t_ldt_RT
  [27] t_ldt_Z
  [28] t_ldt_acc
  [29] t_name_RT
  [30] t_name_Z
  [31] t_name_acc
  [32] relation1_f-t
  [33] relation2_f-t
  [34] FAS_f-t
  [35] BAS_f-t
  [36] f_CUEfanout
  [37] TARGETfanin
  [38] rank
  [39] LSA_f-t
  [40] BEAGLE_RPs
  [41] BEAGLE_PMI
  [42] BEAGLE cosine
  [43] BEAGLE WIKI NIN
  [44] BEAGLE WIKI Cosine


In [8]:
# Cell 6 — Discover the LDT item analysis file
LDT_PATH = DATA_DIR / "raw" / "ldt word item analysis.xlsx"

xls_ldt = pd.ExcelFile(LDT_PATH)
print(f"File: {LDT_PATH.name}")
print(f"Number of sheets: {len(xls_ldt.sheet_names)}")
print(f"\nSheets:")
for i, name in enumerate(xls_ldt.sheet_names):
    n_rows = len(pd.read_excel(LDT_PATH, sheet_name=name, usecols=[0]))
    n_cols = pd.read_excel(LDT_PATH, sheet_name=name, nrows=0).shape[1]
    print(f"  [{i}] {name!r:<35}  {n_rows:>6,} rows x {n_cols:>3} cols")

File: ldt word item analysis.xlsx
Number of sheets: 1

Sheets:
  [0] 'ldt word item analysis'              1,661 rows x  63 cols


In [9]:
# Cell 7 — Load and inspect the LDT item analysis sheet
ldt_raw = pd.read_excel(LDT_PATH, sheet_name="ldt word item analysis")

print(f"Shape: {ldt_raw.shape}")
print(f"\nAll {len(ldt_raw.columns)} columns:")
for i, col in enumerate(ldt_raw.columns):
    print(f"  [{i:>2}] {col}")

Shape: (1661, 63)

All 63 columns:
  [ 0] target
  [ 1] short_1st_rel
  [ 2] short_1st_rel_err
  [ 3] short_1st_un
  [ 4] short_1st_un_err
  [ 5] short_other_rel
  [ 6] short_other_rel_err
  [ 7] short_other_un
  [ 8] short_other_un_err
  [ 9] long_1st_rel
  [10] long_1st_rel_err
  [11] long_1st_un
  [12] long_1st_un_err
  [13] long_other_rel
  [14] long_other_rel_err
  [15] long_other_un
  [16] long_other_un_err
  [17] short_first_priming
  [18] short_other_priming
  [19] long_first_priming
  [20] long_other_priming
  [21] first_priming_overall
  [22] other_priming_overall
  [23] VAR00003
  [24] VAR00001
  [25] prime_1st_assoc
  [26] target_length
  [27] target_loghal
  [28] target_subtitle
  [29] target_logsubtitle
  [30] target_ortho
  [31] target_pos
  [32] target_elprt
  [33] target_elpz
  [34] target_elpacc
  [35] target_fanin
  [36] firstassoc_logsubtitle
  [37] firstassoc_length
  [38] firstassoc_loghal
  [39] firstassoc_ortho
  [40] firstassoc_elp
  [41] firstassoc_relationtyp

In [10]:
# Cell 8 — Extract the prime, target, and priming z-score
spp = ldt_raw[["prime_1st_assoc", "target", "short_first_priming"]].copy()
spp.columns = ["prime", "target", "priming_z"]

print(f"Shape: {spp.shape}")
print(f"\nFirst 10 rows:")
print(spp.head(10).to_string(index=False))
print(f"\nMissing values:")
print(spp.isna().sum())
print(f"\nPriming z-score distribution:")
print(spp["priming_z"].describe())

Shape: (1661, 3)

First 10 rows:
     prime   target  priming_z
    DISOWN  abandon   0.058553
CAPABILITY  ability  -0.063664
    NORMAL abnormal   0.344848
     BELOW    above  -0.192987
       USE    abuse   0.440475
    REJECT   accept   0.273060
  CHECKING  account  -0.063222
     BLAME   accuse  -0.456199
   STOMACH     ache   0.312204
      GOAL  achieve   0.647149

Missing values:
prime        0
target       0
priming_z    0
dtype: int64

Priming z-score distribution:
count    1661.000000
mean        0.209432
std         0.296249
min        -0.568574
25%         0.004440
50%         0.200557
75%         0.395723
max         1.333212
Name: priming_z, dtype: float64


In [11]:
# Cell 9 — Normalize case and audit graph coverage
spp["prime"] = spp["prime"].str.lower().str.strip()
spp["target"] = spp["target"].str.lower().str.strip()

# Confirm the case change worked
print("After lowercasing — first 5 rows:")
print(spp.head().to_string(index=False))

# Build the set of graph node names once for fast lookup
nodes = set(G.nodes())

# For each pair, check whether each word is in the graph
spp["prime_in_graph"] = spp["prime"].isin(nodes)
spp["target_in_graph"] = spp["target"].isin(nodes)
spp["both_in_graph"] = spp["prime_in_graph"] & spp["target_in_graph"]

# Bucket counts
total = len(spp)
both = spp["both_in_graph"].sum()
prime_only_missing = (~spp["prime_in_graph"] & spp["target_in_graph"]).sum()
target_only_missing = (spp["prime_in_graph"] & ~spp["target_in_graph"]).sum()
both_missing = (~spp["prime_in_graph"] & ~spp["target_in_graph"]).sum()

print(f"\nCoverage audit:")
print(f"  Total SPP pairs:        {total:,}")
print(f"  Both words in graph:    {both:,} ({100*both/total:.1f}%)")
print(f"  Prime missing only:     {prime_only_missing:,} ({100*prime_only_missing/total:.1f}%)")
print(f"  Target missing only:    {target_only_missing:,} ({100*target_only_missing/total:.1f}%)")
print(f"  Both missing:           {both_missing:,} ({100*both_missing/total:.1f}%)")

After lowercasing — first 5 rows:
     prime   target  priming_z
    disown  abandon   0.058553
capability  ability  -0.063664
    normal abnormal   0.344848
     below    above  -0.192987
       use    abuse   0.440475

Coverage audit:
  Total SPP pairs:        1,661
  Both words in graph:    1,648 (99.2%)
  Prime missing only:     10 (0.6%)
  Target missing only:    3 (0.2%)
  Both missing:           0 (0.0%)


In [12]:
# Cell 10 — Inspect the 13 missing pairs, then filter
print("Pairs where the prime is missing from the graph:")
missing_prime = spp.loc[~spp["prime_in_graph"], ["prime", "target", "priming_z"]]
print(missing_prime.to_string(index=False))

print(f"\nPairs where the target is missing from the graph:")
missing_target = spp.loc[~spp["target_in_graph"], ["prime", "target", "priming_z"]]
print(missing_target.to_string(index=False))

# Keep only pairs where both words exist
spp_clean = spp.loc[spp["both_in_graph"]].copy().reset_index(drop=True)
print(f"\nAnalyzable pairs: {len(spp_clean):,} (dropped {len(spp) - len(spp_clean)})")

Pairs where the prime is missing from the graph:
      prime      target  priming_z
     clorox      bleach   0.389913
    cowgirl      cowboy   0.232635
    bandaid         cut  -0.005053
elimination destruction   0.123636
    implode     explode   0.136463
earnestness     honesty  -0.067665
       hula        hoop   0.569108
      folly     mistake   0.685312
      frisk      search  -0.049349
     excise         tax  -0.032555

Pairs where the target is missing from the graph:
   prime   target  priming_z
finished    don't   0.263484
  pastry doughnut   0.461826
    hoop     hula   0.580281

Analyzable pairs: 1,648 (dropped 13)


In [13]:
# Cell 11 — Identify response-only primes (in graph but zero outgoing edges)
spp_clean["prime_out_degree"] = spp_clean["prime"].apply(lambda w: G.out_degree(w))
spp_clean["target_out_degree"] = spp_clean["target"].apply(lambda w: G.out_degree(w))

# Response-only = in graph but no outgoing edges
prime_response_only = spp_clean["prime_out_degree"] == 0
target_response_only = spp_clean["target_out_degree"] == 0

print(f"Out of {len(spp_clean):,} analyzable pairs:")
print(f"  Prime has zero outgoing edges (response-only):  "
      f"{prime_response_only.sum():,} ({100*prime_response_only.mean():.1f}%)")
print(f"  Target has zero outgoing edges (response-only): "
      f"{target_response_only.sum():,} ({100*target_response_only.mean():.1f}%)")

# Show some examples
print(f"\nFirst 10 response-only primes (sample):")
sample = spp_clean.loc[prime_response_only, ["prime", "target", "priming_z"]].head(10)
print(sample.to_string(index=False))

# Distribution of prime out-degrees (just the bottom of the distribution matters)
print(f"\nPrime out-degree distribution:")
print(spp_clean["prime_out_degree"].describe(percentiles=[0.05, 0.10, 0.25, 0.50]))

Out of 1,648 analyzable pairs:
  Prime has zero outgoing edges (response-only):  486 (29.5%)
  Target has zero outgoing edges (response-only): 462 (28.0%)

First 10 response-only primes (sample):
    prime    target  priming_z
   disown   abandon   0.058553
     goal   achieve   0.647149
    hyper    active   0.061392
 explorer adventure   0.364770
continent    africa  -0.110911
      for   against   0.068070
 contract agreement  -0.033306
      fan       air  -0.568574
   flying  airplane   0.017782
    drunk   alcohol   0.712113

Prime out-degree distribution:
count    1648.000000
mean       23.127427
std        15.837119
min         0.000000
5%          0.000000
10%         0.000000
25%         0.000000
50%        30.000000
max        49.000000
Name: prime_out_degree, dtype: float64


In [14]:
# Cell 12 — Build undirected graph with summed reciprocal weights and inverted distances
G_undir = nx.Graph()
G_undir.add_nodes_from(G.nodes())

for u, v, data in G.edges(data=True):
    w = data["R123.Strength"]
    if G_undir.has_edge(u, v):
        # Reciprocal directed edge already added — sum the weights
        G_undir[u][v]["weight"] += w
    else:
        G_undir.add_edge(u, v, weight=w)

# Add inverted distance attribute to every undirected edge
for u, v, data in G_undir.edges(data=True):
    data["distance"] = 1.0 / data["weight"]

# Report structural stats
n_directed_edges = G.number_of_edges()
n_undirected_edges = G_undir.number_of_edges()
n_reciprocal = n_directed_edges - n_undirected_edges

print(f"Directed graph:     {G.number_of_nodes():,} nodes, {n_directed_edges:,} directed edges")
print(f"Undirected graph:   {G_undir.number_of_nodes():,} nodes, {n_undirected_edges:,} undirected edges")
print(f"Reciprocal pairs combined: {n_reciprocal:,} "
      f"({100*n_reciprocal/n_directed_edges:.1f}% of directed edges)")

# Sanity check: undirected distances for well-known pairs
print(f"\nSanity check — undirected distances for known pairs:")
test_pairs = [("ocean", "sea"), ("ocean", "water"), ("cat", "dog"),
              ("bread", "butter"), ("doctor", "nurse"), ("hot", "cold"),
              ("spoon", "volcano")]  # last one should be far
for u, v in test_pairs:
    if u not in G_undir or v not in G_undir:
        print(f"  {u:<10} <-> {v:<10}  one or both not in graph")
        continue
    if G_undir.has_edge(u, v):
        d = G_undir[u][v]["distance"]
        w = G_undir[u][v]["weight"]
        print(f"  {u:<10} <-> {v:<10}  direct edge   weight={w:.4f}  distance={d:.2f}")
    else:
        try:
            d = nx.shortest_path_length(G_undir, u, v, weight="distance")
            path = nx.shortest_path(G_undir, u, v, weight="distance")
            print(f"  {u:<10} <-> {v:<10}  multi-hop     distance={d:.2f}  path={' → '.join(path)}")
        except nx.NetworkXNoPath:
            print(f"  {u:<10} <-> {v:<10}  NO PATH")

Directed graph:     24,657 nodes, 279,410 directed edges
Undirected graph:   24,657 nodes, 258,299 undirected edges
Reciprocal pairs combined: 21,111 (7.6% of directed edges)

Sanity check — undirected distances for known pairs:
  ocean      <-> sea         direct edge   weight=0.3554  distance=2.81
  ocean      <-> water       direct edge   weight=0.1555  distance=6.43
  cat        <-> dog         direct edge   weight=0.2047  distance=4.89
  bread      <-> butter      direct edge   weight=0.1976  distance=5.06
  doctor     <-> nurse       direct edge   weight=0.1599  distance=6.26
  hot        <-> cold        multi-hop     distance=11.57  path=hot → temperature → cold
  spoon      <-> volcano     multi-hop     distance=32.58  path=spoon → ladle → soup → hot → lava → volcano


In [15]:
# Cell 13 — Compute undirected shortest-path distances for all 1,648 pairs
from collections import defaultdict
import time

# Group targets by source so each Dijkstra call covers all that source's targets
pairs = list(zip(spp_clean["prime"], spp_clean["target"]))
targets_by_source = defaultdict(list)
for idx, (src, tgt) in enumerate(pairs):
    targets_by_source[src].append((idx, tgt))

distances = [np.nan] * len(pairs)
unreachable_count = 0

start = time.time()
for src, target_list in targets_by_source.items():
    # Single-source Dijkstra returns distances to ALL reachable nodes in one call
    sssp = nx.single_source_dijkstra_path_length(G_undir, src, weight="distance")
    for idx, tgt in target_list:
        if tgt in sssp:
            distances[idx] = sssp[tgt]
        else:
            unreachable_count += 1
elapsed = time.time() - start

spp_clean["dist_undirected"] = distances

print(f"Computed {len(pairs):,} undirected distances in {elapsed:.1f} seconds")
print(f"Unique source nodes (Dijkstra calls): {len(targets_by_source):,}")
print(f"Unreachable pairs: {unreachable_count}")
print(f"\nDistance distribution:")
print(spp_clean["dist_undirected"].describe())

print(f"\nFirst 10 pairs with their distances and priming z-scores:")
print(spp_clean[["prime", "target", "dist_undirected", "priming_z"]].head(10).to_string(index=False))

Computed 1,648 undirected distances in 280.4 seconds
Unique source nodes (Dijkstra calls): 1,648
Unreachable pairs: 0

Distance distribution:
count    1648.000000
mean       19.779349
std        26.115505
min         1.979086
25%         5.110856
50%         9.447924
75%        23.495690
max       225.624855
Name: dist_undirected, dtype: float64

First 10 pairs with their distances and priming z-scores:
     prime   target  dist_undirected  priming_z
    disown  abandon       116.385442   0.058553
capability  ability         6.977247  -0.063664
    normal abnormal         9.458186   0.344848
     below    above         4.640279  -0.192987
       use    abuse        16.193997   0.440475
    reject   accept        21.602462   0.273060
  checking  account         6.983915  -0.063222
     blame   accuse         6.838876  -0.456199
   stomach     ache         6.865627   0.312204
      goal  achieve        14.300000   0.647149
